In [1]:
from loqs.backends import QSimQuantumState, PyGSTiNoiseModel
from loqs.core import QuantumProgram

from loqs.codepacks import d3_5q_code

import pygsti

In [2]:
code_5q = d3_5q_code.create_qec_code()

In [3]:
code_5q.instructions.keys()

dict_keys(['Non-FT Minus Prep', 'FT Minus Prep', 'X', 'Z', 'H', 'Non-FT Physical Z Measure', 'Non-FT Minus Unprep', 'Adaptive Measure Termination', 'Adaptive Measure Part III', 'Adaptive Measure Part II', 'Adaptive Measure Part I', 'Adaptive Measure'])

In [4]:
# Define a pyGSTi processor spec and noise model
# This is the only pyGSTi required here,
# and could eventually be traded out for something else
qubits = ["A0", "A1"] + [f"D{i+2}" for i in range(5)]
gate_names = ["Gxpi", "Gypi", "Gzpi", "Gh", "Gcnot", "Gcphase"]

# TODO: Currently Iz does not need to be set here
# This is because QSimQuantumState actually does not try to pull the rep
# Otherwise, this would result in a KeyError in PyGSTiNoiseModel.get_reps(),
# since it technically should be provided
pspec = pygsti.processors.QubitProcessorSpec(
    len(qubits), 
    gate_names=gate_names,
    qubit_labels=qubits,
    availability={k: "all-permutations" for k in gate_names}
)

ideal_model_pygsti = pygsti.models.create_crosstalk_free_model(pspec)

ideal_model = PyGSTiNoiseModel(ideal_model_pygsti)


In [5]:
# Logical quantum program information
patch_types = {"5Q": code_5q}

# Stack
# The "Init *" instructions will be generated by the QuantumProgram based on the values
# of state_type and patch_types below
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_tyep
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("H", "L0"),
    ("Non-FT Physical Z Measure", "L0")
]

program = QuantumProgram(
    stack,
    default_noise_model=ideal_model,
    state_type=QSimQuantumState,
    patch_types=patch_types,
    name="Test program"
)

In [6]:
# You can do a dry-run, which skips all the apply_fn
# parts of most instructions but otherwise runs the
# whole program through to ensure that args/kwargs
# are available as desired
# I decided to go this route rather than checking
# input/outputs will match everywhere with the hope
# that this will be more efficient. You can do one dry-run
# once taking the actual codepaths through the workflow,
# and then avoid all checks for that program
program.run(dry_run=True)

# If you were to look at the output history,
# you would see "DRY_RUN" for entries like the state
# and measurement_outcomes.
# TBD how I will do these for feedforward operations,
# perhaps just provide a default to use for processing in that case

Executing program run 1
Working on frame 1 (Init State)
Working on frame 2 (Init Patch 5Q)
Working on frame 3 (Non-FT Minus Prep)
Working on frame 4 (H)
Working on frame 5 (Logical H circuit)
Working on frame 6 (Logical H permutation)
Working on frame 7 (Non-FT Physical Z Measure)
Dry run completed successfully!


/Users/sserita/Documents/repos/loqs-public/loqs/core/instructions/instruction.py:94: UserWarning: Skipping param priority for kwargs
  warnings.warn(f"Skipping param priority for {key}")


In [7]:
# And now we can run it for real!
program.run()

Executing program run 2
Working on frame 1 (Init State)
Working on frame 2 (Init Patch 5Q)
Working on frame 3 (Non-FT Minus Prep)
Working on frame 4 (H)
Working on frame 5 (Logical H circuit)
Working on frame 6 (Logical H permutation)
Working on frame 7 (Non-FT Physical Z Measure)


In [8]:
# We can get a pretty verbose output by looking at the history
print(program.run_histories[0])

# Note that the state for everything but the final frame
# is listed as EXPIRED.
# This is my way of indicating that this is a single object that we
# are not storing copies of at each frame. Which keys expire is a
# settable thing in the History object, although an advanced usage

History with 7 items:
  Frame QSimQuantumState state builder result (3 objects):
    'state': EXPIRED
    'instruction': 
      Instruction QSimQuantumState state builder
        Apply signature:(**kwargs) -> 'Frame'
        Data:
          frame_key: state
          obj_class: <class 'loqs.backends.state.qsimstate.QSimQuantumState'>
        Fault-tolerant: True
    'stack': 
      InstructionStack with 4 items:
        InstructionLabel(Init Patch 5Q,None,('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']),{})
        InstructionLabel(Non-FT Minus Prep,L0,(),{})
        InstructionLabel(H,L0,(),{})
        InstructionLabel(Non-FT Physical Z Measure,L0,(),{})

  Frame 5Q patch builder result (4 objects):
    'patches': 
      PatchDict({'L0': 'QECCodePatch for Perfect [[5,1,3]] code on qubits [A0,...,D3]'})
    'instruction': 
      Instruction 5Q patch builder
        Apply signature:(patch_label: 'str', qubits: 'Sequence[str]', qec_code: 'QECCode', patches: 'PatchDict') -> 'Frame'
    

In [9]:
# A good example of the dry run is looking at the last frame for both runs
print("Dry run final frame")
print(program.run_histories[0][-1])

print("\nReal run final frame")
print("Dry run final frame")
print(program.run_histories[1][-1])

# You'll note that things like the stack and patches were done faithfully
# but things like state and measurement_outcomes are not computed
# in the dry run

Dry run final frame
Frame Z-basis measurement for physical qubits result (5 objects):
  'state': 
    DRY_RUN
  'measurement_outcomes': 
    DRY_RUN
  'instruction': 
    Instruction Z-basis measurement for physical qubits
      Apply signature:(model: 'BaseNoiseModel', circuit: 'BasePhysicalCircuit', state: 'BaseQuantumState', include_outcomes: 'bool', inplace: 'bool', reset_mcms: 'bool') -> 'Frame'
      Data:
        include_outcomes: True
        inplace: True
        reset_mcms: False
        circuit: Physical pyGSTi circuit:
          Qubit A0 ---|Iz|---
          Qubit A1 ---|Iz|---
          Qubit D5 ---|Iz|---
          Qubit D2 ---|Iz|---
          Qubit D4 ---|Iz|---
          Qubit D6 ---|Iz|---
          Qubit D3 ---|Iz|---
      Fault-tolerant: False
  'stack': 
    Empty InstructionStack
  'patches': 
    PatchDict({'L0': 'QECCodePatch for Perfect [[5,1,3]] code on qubits [A0,...,D3]'})


Real run final frame
Dry run final frame
Frame Z-basis measurement for physical qub

In [10]:
# We can also run some other programs
prep_unprep_stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_tyep
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("Non-FT Minus Unprep", "L0"),
    ("Non-FT Physical Z Measure", "L0")
]

# We can create a copy of our old program with the stack changed out
# We can also change noise model, state backend, patches, etc.
prep_unprep_program = QuantumProgram.from_quantum_program(program, prep_unprep_stack, name="Prep/unprep test")

In [11]:
prep_unprep_program.run()

Executing program run 1
Working on frame 1 (Init State)
Working on frame 2 (Init Patch 5Q)
Working on frame 3 (Non-FT Minus Prep)
Working on frame 4 (Non-FT Minus Unprep)
Working on frame 5 (Non-FT Physical Z Measure)


In [12]:
print(prep_unprep_program.run_histories[0][-1]["measurement_outcomes"])

MeasurementOutcomes({'A0': [0], 'A1': [0], 'D2': [0], 'D3': [0], 'D4': [0], 'D5': [0], 'D6': [0]})


In [13]:
# Also showcasing the adaptive measurement with feed forward
adaptive_stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_tyep
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("Adaptive Measure", "L0")
]

# We can create a copy of our old program with the stack changed out
# We can also change noise model, state backend, patches, etc.
adaptive_program = QuantumProgram.from_quantum_program(program, adaptive_stack, name="Adaptive test")

In [14]:
# In this run, we ran all the way through the adaptive
# Something is wrong my implementation of measurements?
# Or my prep circuit?
# But the feed forward is working, just getting bad data!
# TODO: It's possible my test is bad!
adaptive_program.run()

Executing program run 1
Working on frame 1 (Init State)
Working on frame 2 (Init Patch 5Q)
Working on frame 3 (Non-FT Minus Prep)
Working on frame 4 (Adaptive Measure)
Working on frame 5 (Adaptive Measure Part II)
Working on frame 6 (Adaptive Measure Part III)
Working on frame 7 (Non-FT Minus Unprep)


In [15]:
print(adaptive_program.run_histories[0])

History with 7 items:
  Frame QSimQuantumState state builder result (3 objects):
    'state': EXPIRED
    'instruction': 
      Instruction QSimQuantumState state builder
        Apply signature:(**kwargs) -> 'Frame'
        Data:
          frame_key: state
          obj_class: <class 'loqs.backends.state.qsimstate.QSimQuantumState'>
        Fault-tolerant: True
    'stack': 
      InstructionStack with 3 items:
        InstructionLabel(Init Patch 5Q,None,('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']),{})
        InstructionLabel(Non-FT Minus Prep,L0,(),{})
        InstructionLabel(Adaptive Measure,L0,(),{})

  Frame 5Q patch builder result (4 objects):
    'patches': 
      PatchDict({'L0': 'QECCodePatch for Perfect [[5,1,3]] code on qubits [A0,...,D6]'})
    'instruction': 
      Instruction 5Q patch builder
        Apply signature:(patch_label: 'str', qubits: 'Sequence[str]', qec_code: 'QECCode', patches: 'PatchDict') -> 'Frame'
        Data:
          qec_code: QECCode Perfect 